# Часть 2. Обогащение данных из стороннего датасета

В этом ноутбуке мы загружаем очищенный датасет `netflix_clean_1.csv`.
И обогащаем его информацией о жанрах, бюджете, выручке, хронометраже, данные об аудитории и оценке из внешних источников.

Результат этой части - это обогащенный датасет `netflix_enriched_2.csv`, который используется в последующих частях анализа.

#### Выполнили Соломонов Егор, Марокина Татьяна


## Шаг 1. Импорт библиотек и чтение датасетов


In [ ]:
import pandas as pd
import numpy as np
import ast

### 1.1 Чтение "очищенного" датасета Netflix


In [ ]:
df_cleared = pd.read_csv("netflix_clean_1.csv")
print(df_cleared.shape)
df_cleared.head(3)

### 1.2 Чтение стороннего датасета `TMDB_IMDB_Movies_Dataset.csv`

Датасет взят с [kaggle](https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset?select=TMDB++IMDB+Movies+Dataset.csv). Датаесет содержит 437246 строк и 29 признаков.


In [ ]:
df_tmdb_imdb_movies = pd.read_csv("TMDB_IMDB_Movies_Dataset.csv")
print(df_tmdb_imdb_movies.shape)
df_tmdb_imdb_movies.head(3)

### 1.3 Чтение стороннего датасета `movies_metadata.csv`

Датасет взят с [kaggle](https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset?select=movies_metadata.csv). Датаесет содержить 45466 строк и 24 признака.


In [ ]:
df_mm = pd.read_csv("movies_metadata.csv", low_memory=False)
print(df_mm.shape)
df_mm.head(3)

## Шаг 2. Чистка датасетов и подготовка к слиянию


### 2.1 Подготовка `df_cleared`

Приготовим к слиянию датасет `df_cleared`.

- Убираем дубли по заголовку `title` и году выпуска `release_year`
- Приводим год выпуска `release_year` к числу для дальнейшего сравнения
- Заголовок `title` приводим к нижнему регистру для точного сравнения


In [ ]:
df_cleared["title_lower"] = df_cleared["title"].str.lower().str.strip()
df_cleared["release_year"] = df_cleared["release_year"].astype("Int64")

df_cleared = df_cleared.rename(
    columns={
        "rating_score": "rating_score_nfx",
        "rating_size": "rating_size_nfx",
    }
)

df_cleared = df_cleared.drop_duplicates(subset=["title", "release_year"]).reset_index(
    drop=True
)

print(df_cleared.shape)
df_cleared.head(3)

### 2.2 Чистка `df_mm` и `df_tmdb_imdb_movies`

Приготовим к слиянию датасеты. Убираем лишние признаки, которые не планируем использовать в анализе


In [ ]:
columns = [
    "title",
    "imdb_id",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "vote_average",
    "vote_count",
    "release_date",
]
df_mm = df_mm[columns].copy()

columns = [
    "title",
    "tconst",
    "budget",
    "revenue",
    "runtime",
    "genres",
    "vote_average",
    "vote_count",
    "release_date",
    "averageRating",
    "numVotes",
]
df_tmdb_imdb_movies = df_tmdb_imdb_movies[columns].copy()


Для избежания ошибок в слиянии датасетов необходимо избавиться полностью от строк без названия. Все остальные признаки можно пока оставить как есть, поскольку избавляться от пропусков, если они есть, мы должны будем в изначальном датасете. Пока-что мы просто пытаемся обогатить его информацией

- Избавимся также от всех пропусков и дублей по полю `title`. Для целей обогащения такие строки попросту не нужны.
- Дополнительно подготовим столбец с годом `release_year`, для дальнейшего слияния в паре с названием `title`.
- переименуем столбцы для единообразия и разделения источников (для рейтинга)


In [ ]:
df_mm = df_mm.dropna(subset=["title"]).copy()
df_mm["title"] = df_mm["title"].str.lower()

df_mm = df_mm.rename(
    columns={"vote_average": "vote_average_tmdb"}
)
df_mm = df_mm.rename(
    columns={"vote_count": "vote_count_tmdb"}
)

df_mm["release_year"] = pd.to_datetime(df_mm["release_date"], errors="coerce").dt.year
df_mm["release_year"] = df_mm["release_year"].astype("Int64")
df_mm = (
    df_mm.drop(columns=["release_date"])
    .drop_duplicates(subset=["title", "release_year"])
    .reset_index(drop=True)
)

print(df_mm.shape)
df_mm.head(3)

In [ ]:
df_tmdb_imdb_movies = df_tmdb_imdb_movies.dropna(subset=["title"]).copy()
df_tmdb_imdb_movies["title"] = df_tmdb_imdb_movies["title"].str.lower()

df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(columns={"tconst": "imdb_id"})
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"averageRating": "vote_average_imdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"numVotes": "vote_count_imdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"vote_average": "vote_average_tmdb"}
)
df_tmdb_imdb_movies = df_tmdb_imdb_movies.rename(
    columns={"vote_count": "vote_count_tmdb"}
)

df_tmdb_imdb_movies["release_year"] = pd.to_datetime(
    df_tmdb_imdb_movies["release_date"], errors="coerce"
).dt.year
df_tmdb_imdb_movies["release_year"] = df_tmdb_imdb_movies["release_year"].astype(
    "Int64"
)
df_tmdb_imdb_movies = (
    df_tmdb_imdb_movies.drop(columns=["release_date"])
    .drop_duplicates(subset=["title", "release_year"])
    .reset_index(drop=True)
)

print(df_tmdb_imdb_movies.shape)
df_tmdb_imdb_movies.head(3)

Также можно сразу заметить одну странность: в иных строках `budget` и `revenue` отмечено 0. Это явный пропуск, поскольку производство фильма не может стоить 0 и принести 0 дохода


Избавимся от пропусков по полям `budget` и `revenue`


In [ ]:
df_mm["budget"] = df_mm["budget"].apply(lambda x: np.nan if int(x) == 0 else int(x))
df_mm["revenue"] = df_mm["revenue"].apply(lambda x: np.nan if int(x) == 0 else int(x))

df_tmdb_imdb_movies["budget"] = df_tmdb_imdb_movies["budget"].apply(
    lambda x: np.nan if int(x) == 0 else int(x)
)
df_tmdb_imdb_movies["revenue"] = df_tmdb_imdb_movies["revenue"].apply(
    lambda x: np.nan if int(x) == 0 else int(x)
)

In [ ]:
df_mm.isna().sum()

In [ ]:
df_tmdb_imdb_movies.isna().sum()

Пропусков слишком много (почти 70 % датасета) — их нельзя заменить медианой или средним, данных слишком мало. В данный момент лучше ничего с ними не делать и провести слияние датасетов и потом оценить результат слияния


Преобразуем поле `genres` в датасете `df_mm`. Стоит отметить, что оно записано не в виде JSON, а в форме простого python literal.

ПРиведем его к виду - строка с жанрами через запятую и пробел, как это записанов датасете `df_tmdb_imdb_movies`


In [ ]:
def extract_name(x: str, field: str) -> str | float:
    if pd.isna(x) or x == "":
        return np.nan
    try:
        genres_json = ast.literal_eval(x)
        result = []

        for genre in genres_json:
            if isinstance(genre, dict) and field in genre:
                result.append(genre[field])

        return ", ".join(result) if result else np.nan

    except:
        return np.nan


df_mm["genres"] = df_mm["genres"].apply(lambda x: extract_name(str(x), "name"))

In [ ]:
df_mm.isna().sum()

In [ ]:
df_mm.head(3)

## Шаг 3. Слияние датасетов


Будем поочередно мерджить исходный датасет `df_cleared` сначала с датасетом `df_tmdb_imdb_movies`, затем с датасетом `df_mm`, чтобы максимально получить новых данных и сократить количество возможных пропусков.


### 3.1 Мерджим `df_cleared` с `df_tmdb_imdb_movies` по `title` + `release_year`

Слияние будем выполнять по заголовку `title` и году выпуска контента `release_year`


In [ ]:
df_cleared_copy = df_cleared.copy()

df_merged = pd.merge(
    df_cleared_copy,
    df_tmdb_imdb_movies,
    left_on=["title_lower", "release_year"],
    right_on=["title", "release_year"],
    how="left",
    indicator=True,
)
print(df_merged.shape)
df_merged.head(3)

Убираем лишние столбцы после мерджа и переименовываем оставшиеся. И проверяем сколько нашлось совпадений


In [ ]:
df_merged = df_merged.rename(
    columns={"title_x": "title", "release_year_x": "release_year"}
)
df_merged["_merge_tmdb"] = df_merged["_merge"]
df_merged = df_merged.drop(columns=["_merge", "title_y"])

matched_count = (df_merged["_merge_tmdb"] == "both").sum()
print(f"3.1: смерджено {matched_count} записей по title + year")
df_merged.head(3)

Проверим, что одинаковые названия но разные года выпуска смержены корректно


In [ ]:
duplicated_titles_list = df_merged["title_lower"][
    df_merged.duplicated(subset=["title_lower"], keep=False)
].unique()
print(duplicated_titles_list)

In [ ]:
df_tmdb_imdb_movies[df_tmdb_imdb_movies["title"] == "skins"]

In [ ]:
df_merged[df_merged["title_lower"] == "skins"]

In [ ]:
df_merged.isna().sum()

### 3.2 Мерджим незаполненные в `df_merged` с `df_tmdb_imdb_movies` по `title` + `release_year` в диапазоне +- один год

В ходе работы с датасетами обнаружилось, что встречаются одинаковые фильмы по названию, но год не совпадает с разницей в +- один год. Сложно ответить, с чем это связано.

Поэтому слияние будем выполнять также по заголовку `title` и по году выпуска контента `release_year` +- один год

Выполним это только для строк, которые не были смерджены ранее `_merge_tmdb` == `left_only`


In [ ]:
df_merged[df_merged["title_lower"] == "lottie dottie chicken"]

In [ ]:
df_tmdb_imdb_movies[df_tmdb_imdb_movies["title"] == "lottie dottie chicken"]

In [ ]:
df_with_tmdb = df_merged[df_merged["_merge_tmdb"] == "both"].copy()
df_without_tmdb = df_merged[df_merged["_merge_tmdb"] == "left_only"].copy()

df_without_tmdb_copy = df_without_tmdb.drop(
    columns=[
        "revenue",
        "runtime",
        "_merge_tmdb",
        "imdb_id",
        "budget",
        "genres",
        "vote_average_tmdb",
        "vote_count_tmdb",
        "vote_average_imdb",
        "vote_count_imdb",
    ]
).copy()
print(df_without_tmdb_copy.shape)
df_without_tmdb_copy.head(3)

In [ ]:
df_unmatched_merged = pd.merge(
    df_without_tmdb_copy,
    df_tmdb_imdb_movies,
    left_on="title_lower",
    right_on="title",
    how="left",
    indicator=True,
    suffixes=("_netflix", "_tmdb"),
)

# Проверяем диапазон года (+-1 год от release_year)
year_in_range = df_unmatched_merged["release_year_tmdb"].isna() | (
    (
        df_unmatched_merged["release_year_tmdb"]
        >= df_unmatched_merged["release_year_netflix"] - 1
    )
    & (
        df_unmatched_merged["release_year_tmdb"]
        <= df_unmatched_merged["release_year_netflix"] + 1
    )
)

# То, что смерджилось но не в диапазоне, заполняем NaN
df_unmatched_merged.loc[
    ~year_in_range,
    [
        "revenue",
        "runtime",
        "imdb_id",
        "budget",
        "genres",
        "vote_average_tmdb",
        "vote_count_tmdb",
        "vote_average_imdb",
        "vote_count_imdb",
    ],
] = np.nan


df_unmatched_merged = df_unmatched_merged.drop(
    columns=["title_tmdb", "release_year_tmdb"]
)
df_unmatched_merged = df_unmatched_merged.rename(
    columns={"title_netflix": "title", "release_year_netflix": "release_year"}
)

df_unmatched_merged["_merge_tmdb"] = df_unmatched_merged["_merge"]
df_unmatched_merged = df_unmatched_merged.drop(columns=["_merge"])

matched_count = (
    (df_unmatched_merged["_merge_tmdb"] == "both")
    & (df_unmatched_merged["vote_average_tmdb"].notna())
).sum()
print(
    f"3.2: смерджено еще {matched_count} записей по title с годом в диапазоне +-1 год"
)

df_unmatched_merged = df_unmatched_merged.drop_duplicates(
    subset=["title", "release_year"]
).reset_index(drop=True)
print(df_unmatched_merged.shape)


In [ ]:
df_merged = pd.concat([df_with_tmdb, df_unmatched_merged], ignore_index=True)
df_merged = df_merged.drop(columns=["_merge_tmdb"])
print(df_merged.shape)
df_merged.isna().sum()

### 3.3 Мерджим незаполненные в `df_merged` с `df_mm` по `title` + `release_year` в диапазоне +- один год

Пропуски остались. Возможно, второй датасет позволит заполнить пробелы.

Проведем слияние со вторым датасетом.


In [ ]:
df_with_data = df_merged[df_merged["vote_average_tmdb"].notna()].copy()
df_without_data = df_merged[df_merged["vote_average_tmdb"].isna()].copy()
df_without_data_copy = df_without_data.drop(
    columns=[
        "vote_average_tmdb",
        "vote_count_tmdb",
        "revenue",
        "runtime",
        "imdb_id",
        "budget",
        "genres",
    ]
).copy()
df_without_data_copy.shape

In [ ]:
df_unmatched_merged = pd.merge(
    df_without_data_copy,
    df_mm,
    left_on="title_lower",
    right_on="title",
    how="left",
    indicator=True,
    suffixes=("_netflix", "_mm"),
)

# Проверяем диапазон года (+-1 год от release_year)
year_in_range = df_unmatched_merged["release_year_mm"].isna() | (
    (
        df_unmatched_merged["release_year_mm"]
        >= df_unmatched_merged["release_year_netflix"] - 1
    )
    & (
        df_unmatched_merged["release_year_mm"]
        <= df_unmatched_merged["release_year_netflix"] + 1
    )
)

# То, что смерджилось но не в диапазоне, заполняем NaN
df_unmatched_merged.loc[
    ~year_in_range,
    [
        "vote_average_tmdb",
        "vote_count_tmdb",
        "revenue",
        "runtime",
        "imdb_id",
        "budget",
        "genres",
    ],
] = np.nan


df_unmatched_merged = df_unmatched_merged.drop(columns=["title_mm", "release_year_mm"])
df_unmatched_merged = df_unmatched_merged.rename(
    columns={"title_netflix": "title", "release_year_netflix": "release_year"}
)

df_unmatched_merged["_merge_mm"] = df_unmatched_merged["_merge"]
df_unmatched_merged = df_unmatched_merged.drop(columns=["_merge"])

matched_count = (
    (df_unmatched_merged["_merge_mm"] == "both")
    & (df_unmatched_merged["vote_average_tmdb"].notna())
).sum()
print(
    f"3.3: смерджено еще {matched_count} записей по title с годом в диапазоне +-1 год"
)

df_unmatched_merged = df_unmatched_merged.drop_duplicates(
    subset=["title", "release_year"]
).reset_index(drop=True)
print(df_unmatched_merged.shape)


In [ ]:
df_merged = pd.concat([df_with_data, df_unmatched_merged], ignore_index=True)
df_merged = df_merged.drop(columns=["_merge_mm"])
print(df_merged.shape)


## Шаг 4. Заполнение пропусков в финальном датасете


Проверим, сколько осталось пропусков в результате


In [ ]:
df_merged.isna().sum()

Таким образом, в некоторых полях по-прежнему имеются пропуски.


### 4.1 Заполнение пропусков в `genres`, `runtime` и `imdb_id`

Подключим дополнительны [датасет](https://developer.imdb.com/non-commercial-datasets/?utm_source=chatgpt.com#titlebasicstsvgz) с официального сайта IMDB, который содержит информацию о жанрах, времени контента


In [ ]:
df_imdb = pd.read_csv("title.basics.tsv", sep="\t", na_values=["\\N"], low_memory=False)

In [ ]:
df_imdb.head(3)

Подготовим данные к слиянию

- Определим нужные поля
- Подготовим заголовок для слияния (приведем к нижнему регистру)
- Колонку с жанрами отформатируем как в итоговом датасете


In [ ]:
df_imdb_copy = df_imdb[
    ["tconst", "primaryTitle", "startYear", "endYear", "runtimeMinutes", "genres"]
].copy()

In [ ]:
df_imdb_copy["runtimeMinutes"] = pd.to_numeric(
    df_imdb_copy["runtimeMinutes"], errors="coerce"
).astype("Int64")

df_imdb_copy["title_lower"] = df_imdb_copy["primaryTitle"].str.lower().str.strip()
df_imdb_copy = df_imdb_copy.drop(columns=["primaryTitle"])

df_imdb_copy["genres"] = df_imdb_copy["genres"].str.replace(",", ", ")

print(df_imdb_copy.shape)
df_imdb_copy.head(5)

Выполним слияние по заголовку


In [ ]:
df_merged_imdb = pd.merge(
    df_merged,
    df_imdb_copy,
    left_on="title_lower",
    right_on="title_lower",
    how="left",
    suffixes=("_netflix", "_imdb"),
)


Определим функцию, с помощью которой будем проверять входит ли год release_year в исходном датасете в диапазон годов из датасета IMDB, так как слияние только по заголовкам может быть не точным.

`startYear` или `endYear` должны быть близки к `release_year`. Диапазон как и ранее +- один год.


In [ ]:
def check_year_match(row):
    if pd.isna(row["release_year"]):
        return True

    netflix_year = int(row["release_year"])
    start_year = row["startYear"]
    end_year = row["endYear"]

    if pd.notna(start_year):
        if abs(int(start_year) - netflix_year) <= 1:
            return True

    if pd.notna(end_year):
        if abs(int(end_year) - netflix_year) <= 1:
            return True

    return False


Применить проверку года. И выведем итоговый датасет.


In [ ]:
df_merged_imdb["year_match"] = df_merged_imdb.apply(check_year_match, axis=1)

# Обнулим данные если год не подходит
cols_to_nullify = ["tconst", "runtimeMinutes", "genres_imdb"]
for col in cols_to_nullify:
    if col in df_merged_imdb.columns:
        df_merged_imdb.loc[~df_merged_imdb["year_match"], col] = np.nan

# Заполним пропуски
df_merged_imdb["imdb_id"] = df_merged_imdb["imdb_id"].fillna(df_merged_imdb["tconst"])
df_merged_imdb["runtime"] = df_merged_imdb["runtime"].fillna(
    df_merged_imdb["runtimeMinutes"]
)
df_merged_imdb["genres"] = df_merged_imdb["genres_netflix"].fillna(
    df_merged_imdb["genres_imdb"]
)

df_final = df_merged_imdb.drop(
    columns=[
        "startYear",
        "endYear",
        "tconst",
        "runtimeMinutes",
        "genres_imdb",
        "year_match",
        "genres_netflix",
    ]
)

df_final = df_final.drop_duplicates(subset=["title", "release_year"]).reset_index(
    drop=True
)

print(df_final.shape)

In [ ]:
df_final.isna().sum()

Благодаря датасету IMDB удалось существенно сократить количество пропусков в полях `genres`, `runtime`, `imdb_id`


Заполним пропуски по полю `genres`.
Для этого применим метод Мультиметки по консолидированным жанрам. Такой вариант пригодится нам в последующем анализе.

- Выделим подгруппы жанров,
- добавим соответствующие бинарные столбцы.


In [ ]:
def parse_all_genres(genres_str):
    if pd.isna(genres_str) or genres_str == "":
        return []

    return [g.strip() for g in str(genres_str).split(", ")]


genre_mapper = {
    "Action": "Action",
    "Crime": "Action",
    "Adventure": "Action",
    "Thriller": "Action",
    "Western": "Action",
    "Comedy": "Comedy",
    "Drama": "Drama",
    "Romance": "Drama",
    "Family": "Family",
    "Animation": "Family",
    "Children": "Family",
    "Documentary": "Family",
    "Horror": "Other",
    "Fantasy": "Other",
    "Sci-Fi": "Other",
    "Science Fiction": "Other",
    "Mystery": "Other",
    "Music": "Other",
}

df_final["genres_list"] = df_final["genres"].apply(parse_all_genres)
df_final["genres_consolidated"] = df_final["genres_list"].apply(
    lambda genres_list: list(
        set([genre_mapper.get(g, "Other") for g in genres_list if g])
    )
)

for genre in ["Action", "Comedy", "Drama", "Family", "Other"]:
    df_final[f"is_{genre.lower()}"] = df_final["genres_consolidated"].apply(
        lambda x: 1 if genre in x else 0
    )

Итого распределено по жанрам


In [ ]:
for genre in ["Action", "Comedy", "Drama", "Family", "Other"]:
    count = df_final[f"is_{genre.lower()}"].sum()
    print(f"{genre}: {count}")


Мультиметка реализована с помощью бинарных колонок (one-hot encoding). Каждый фильм может иметь несколько жанров из набора `{Action, Comedy, Drama, Family, Other}`.


In [ ]:
df_final.head(3)

Было решено не заполнять пропуски в столбцах `budget` и `revenue`. Во-первых, они составляют 60 % от всего датасета. Во-вторых, не существует определенной зависимости между ними и другими признаками, или недостаточно данных для ее выявления. Бюджет зависит прежде всего от кинопроизводителя и актерского состава — 500 строк недостаточно, чтобы обучить эффективную модель.


In [ ]:
df_final.corr(numeric_only=True)

In [ ]:
df_final.isna().sum()

In [ ]:
df_final.to_csv("netflix_enriched_2.csv", index=False)